# 03 — Faster R-CNN Training

Run after notebook 01. Loads `config.json` and COCO splits generated earlier.


In [14]:
import os, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or Path('/kaggle').exists()
IS_COLAB = 'COLAB_RELEASE_TAG' in os.environ
WORK_DIR = Path('/kaggle/working') if IS_KAGGLE else (Path('/content') if IS_COLAB else Path.cwd())
cfg = WORK_DIR/'config.json'
if not cfg.exists():
    raise FileNotFoundError(f'Missing config.json at {cfg}. Run notebook 01 first.')
config = json.load(open(cfg))
import torch
print('GPU available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('WARNING: CUDA unavailable; detector training will be slow.')


CUDA OK 2.10.0+cu128


In [15]:
import json, numpy as np, pandas as pd
from pathlib import Path
results_dir = Path(config['results_dir']); results_dir.mkdir(parents=True, exist_ok=True)
dataset_root = Path(config['dataset_root'])
for s in ['train','valid','test']:
    ann = dataset_root/s/'_annotations.coco.json'
    if not ann.exists():
        raise FileNotFoundError(f'Missing annotation file: {ann}')
print('Dataset and annotation checks passed.')


Device: cuda


In [16]:
class RhizomeDetectionDataset(Dataset):
    def __init__(self, ann_file, img_dir, augment=False):
        with open(ann_file) as f:
            coco = json.load(f)
        self.img_dir = img_dir
        self.images  = coco["images"]
        self.ann_map = {}
        for ann in coco["annotations"]:
            self.ann_map.setdefault(ann["image_id"], []).append(ann)
        self.aug = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.10, contrast=0.10),
        ]) if augment else None

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        info = self.images[idx]
        img  = Image.open(os.path.join(self.img_dir, info["file_name"])).convert("RGB")
        if self.aug:
            img = self.aug(img)
        boxes, labels = [], []
        for ann in self.ann_map.get(info["id"], []):
            x, y, w, h = [float(v) for v in ann["bbox"]]
            if w > 2 and h > 2:
                W, H  = img.size
                x1, y1 = max(0, x), max(0, y)
                x2, y2 = min(W, x + w), min(H, y + h)
                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(ann["category_id"])
        boxes  = torch.tensor(boxes,  dtype=torch.float32) if boxes  else torch.zeros((0, 4))
        labels = torch.tensor(labels, dtype=torch.int64)   if labels else torch.zeros((0,), dtype=torch.int64)
        return transforms.ToTensor()(img), {
            "boxes":    boxes,
            "labels":   labels,
            "image_id": torch.tensor([info["id"]]),}

def collate_fn(batch):
    return tuple(zip(*batch))


train_det = RhizomeDetectionDataset(train_ann, train_img_dir, augment=True)
valid_det = RhizomeDetectionDataset(valid_ann, valid_img_dir, augment=False)
test_det  = RhizomeDetectionDataset(test_ann,  test_img_dir,  augment=False)

train_det_loader = DataLoader(train_det, batch_size=4, shuffle=True,  collate_fn=collate_fn, num_workers=0)
valid_det_loader = DataLoader(valid_det, batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=0)
test_det_loader  = DataLoader(test_det,  batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=0)

with open(train_ann) as f:
    info = json.load(f)
NUM_CLASSES = len(info["categories"]) + 1
print("Num classes:", NUM_CLASSES)

rcnn_model = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT,trainable_backbone_layers=3,).to(DEVICE)
in_feat = rcnn_model.roi_heads.box_predictor.cls_score.in_features
rcnn_model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, NUM_CLASSES)
print("Model Initiated")

Num classes: 5
Model Initiated


In [17]:
# Save detector artifacts and thesis metrics
# Replace placeholders below with computed values from training/evaluation cells.
metrics = {'iou': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'map': 0.0, 'miou': 0.0}
import json, pandas as pd
with open(results_dir/'rcnn_metrics.json','w') as f: json.dump(metrics,f,indent=2)
pd.DataFrame([metrics]).to_csv(results_dir/'rcnn_metrics.csv', index=False)
print('Saved:', results_dir/'rcnn_metrics.json')
print('Saved:', results_dir/'rcnn_metrics.csv')
print('Expected additional outputs: rcnn_best.pth and results/rcnn_sample_predictions/')


RuntimeError: Expected all tensors to be on the same device, but got mat1 is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA_addmm)